In [ ]:
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np

from uavloc.data.multidate import MULTIDATE, esri_dates
from uavloc.data.rasters import load_raster
from uavloc.datasets import DERIVED, data_root, dataset_dir, verify_dataset, write_manifest
from uavloc.geo import LatLon
from uavloc.simulation.route import check_inside, random_waypoints, route_length_m, sample_route

# Params

In [ ]:
NAME = "sim-flights"
VERSION = "1"
SOURCE = "ua-village-multidate"
EXCLUDED_VILLAGES = ("birky", "dubrova")   # the segmentation models' training ground
SPLIT_SEED = 42
N_TUNING = 10
ROUTES_PER_VILLAGE = 2
WAYPOINTS_PER_ROUTE = 4
EDGE_MARGIN_M = 80.0
MAX_ATTEMPTS = 20
ALTITUDE_M = 40.0
SPEED_MPS = 10.0
STEP_S = 0.5
CORRECTION_INTERVAL_S = 5.0
FOV_W_DEG = 60.0
DRONE_DATE = "current"
MAP_DATE = "newest"
INS = {"velocity_bias_mps": 0.3, "velocity_random_walk_mps": 0.02, "heading_bias_deg": 2.0, "speed_scale_std": 0.01,
       "reported_yaw_noise_deg": 1.0, "reported_altitude_noise_m": 1.0, "initial_error_m": 0.0}
N_PREVIEW = 6

# Data

In [ ]:
villages = sorted(p.name for p in (DERIVED / MULTIDATE).iterdir()
                  if p.is_dir() and not p.name.startswith(EXCLUDED_VILLAGES))
order = np.random.default_rng(SPLIT_SEED).permutation(len(villages))
split_of = {villages[i]: ("tuning" if rank < N_TUNING else "test") for rank, i in enumerate(order)}
{"villages": len(villages), "tuning": sorted(v for v, s in split_of.items() if s == "tuning")}

In [ ]:
def route_seed(village: str, route_index: int) -> int:
    return int.from_bytes(hashlib.sha256(f"{village}:{route_index}".encode()).digest()[:4], "big")

In [ ]:
def draw_route(village: str, route_index: int, drone_raster, map_raster) -> tuple[list, int]:
    """Waypoints that keep every pose `EDGE_MARGIN_M` inside both rasters, and how many draws it took."""
    rng = np.random.default_rng(route_seed(village, route_index))
    for attempt in range(1, MAX_ATTEMPTS + 1):
        waypoints = random_waypoints(map_raster, WAYPOINTS_PER_ROUTE, EDGE_MARGIN_M, rng)
        poses = sample_route(waypoints, SPEED_MPS, STEP_S, ALTITUDE_M)
        try:
            check_inside(poses, drone_raster, EDGE_MARGIN_M, "drone raster")
            check_inside(poses, map_raster, EDGE_MARGIN_M, "map raster")
        except ValueError:
            continue
        return waypoints, attempt
    raise ValueError(f"{village} route {route_index}: no route inside both rasters in {MAX_ATTEMPTS} draws")

# Run

In [ ]:
routes, attempts, map_dates = [], [], {}
for village in villages:
    drone_raster = load_raster({"kind": "multidate", "place": village, "date": DRONE_DATE})
    map_raster = load_raster({"kind": "multidate", "place": village, "date": MAP_DATE})
    map_dates[village] = esri_dates(village)[-1]
    for route_index in range(ROUTES_PER_VILLAGE):
        waypoints, n = draw_route(village, route_index, drone_raster, map_raster)
        poses = sample_route(waypoints, SPEED_MPS, STEP_S, ALTITUDE_M)
        routes.append({"village": village, "route_index": route_index, "seed": route_seed(village, route_index),
                       "split": split_of[village], "waypoints": [list(w) for w in waypoints],
                       "length_m": route_length_m(poses), "duration_s": poses[-1].t_s})
        attempts.append(n)
len(routes)

In [ ]:
out = data_root(NAME, create=True)
(out / "routes.json").write_text(json.dumps(routes, indent=1))
lengths = np.array([r["length_m"] for r in routes])
{"routes": len(routes), "length_m min/median/max": (lengths.min().round(), np.median(lengths).round(), lengths.max().round())}

# Plots

In [ ]:
fig, axes = plt.subplots(2, N_PREVIEW // 2, figsize=(15, 8))
for ax, route in zip(axes.flat, routes[::len(routes) // N_PREVIEW]):
    raster = load_raster({"kind": "multidate", "place": route["village"], "date": MAP_DATE})
    ax.imshow(raster.pixels)
    xy = np.array([raster.to_px(LatLon(*w)) for w in route["waypoints"]])
    ax.plot(xy[:, 0], xy[:, 1], "-o", color="tab:red", lw=2, ms=4)
    ax.set_title(f"{route['village'][:20]} #{route['route_index']} ({route['split']}, {route['length_m']:.0f} m)", fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(dataset_dir(NAME, create=True) / "preview.png", dpi=80)

# Register

In [ ]:
checks = {
    "routes": len(routes), "villages": len(villages),
    "tuning_villages": sum(s == "tuning" for s in split_of.values()),
    "test_villages": sum(s == "test" for s in split_of.values()),
    "all_inside_both_rasters_at_margin_m": EDGE_MARGIN_M,
    "draws_per_route_max": int(max(attempts)),
    "length_m_median": float(np.median(lengths)), "length_m_min": float(lengths.min()), "length_m_max": float(lengths.max()),
    "fixes_per_route_median": float(np.median([r["duration_s"] // CORRECTION_INTERVAL_S for r in routes])),
}
checks

In [ ]:
write_manifest(
    NAME, version=VERSION,
    description="Random 4-waypoint flight routes over ua-village-multidate villages (birky, dubrova excluded), "
                "2 per village, 10 tuning / rest test by village (seed 42), with the flight and INS settings of 027.",
    source=SOURCE,
    config={"villages": split_of, "map_dates": map_dates, "routes_per_village": ROUTES_PER_VILLAGE,
            "waypoints_per_route": WAYPOINTS_PER_ROUTE, "edge_margin_m": EDGE_MARGIN_M, "split_seed": SPLIT_SEED,
            "n_tuning": N_TUNING, "altitude_m": ALTITUDE_M, "speed_mps": SPEED_MPS, "step_s": STEP_S,
            "correction_interval_s": CORRECTION_INTERVAL_S, "fov_w_deg": FOV_W_DEG, "drone_date": DRONE_DATE,
            "map_date": MAP_DATE, "ins": INS},
    samples=len(routes), checks=checks,
)
verify_dataset(NAME)